In [1]:
import pandas as pd
import numpy as np

In [3]:
# Setting seed
np.random.seed(42)

# Number of participants
participants = 120

# Generate participant IDs
participant_id = np.arange(1, participants + 1)

# Experimental groups
group = np.random.choice(
    ["Control", "Mindfulness"],
    size=participants
)

# Age distribution
age = np.random.normal(
    loc=30,
    scale=8,
    size=participants
).round()

# Stress scores (0–40)
stress_score = np.random.normal(
    loc=22,
    scale=6,
    size=participants
).round(1)

# Sleep quality (1–10)
sleep_quality = np.random.normal(
    loc=6,
    scale=1.5,
    size=participants
).round(1)

# Reaction time in milliseconds
reaction_time = np.random.normal(
    loc=550,
    scale=80,
    size=participants
).round(0)

# Create DataFrame
df = pd.DataFrame({
    "participant_id": participant_id,
    "group": group,
    "age": age,
    "stress_score": stress_score,
    "sleep_quality": sleep_quality,
    "reaction_time_ms": reaction_time
})

# Introduce missing values
for col in ["stress_score", "sleep_quality"]:
    missing_indices = np.random.choice(df.index, size=8, replace=False)
    df.loc[missing_indices, col] = np.nan

# Save dataset
df.to_csv(
    r"F:\doc-hub\education\psychology\projects\computational-cognitive-science-training\data\04-mini-project-raw.csv",
    index=False)

In [6]:
df = pd.read_csv(r"F:\doc-hub\education\psychology\projects\computational-cognitive-science-training\data\04-mini-project-raw.csv")

print(df.head())

   participant_id        group   age  stress_score  sleep_quality  \
0               1      Control  18.0          27.8            9.2   
1               2  Mindfulness  24.0          24.5            3.1   
2               3      Control  26.0          26.9            NaN   
3               4      Control  38.0          33.4            6.9   
4               5      Control  33.0          20.5            6.4   

   reaction_time_ms  
0             514.0  
1             600.0  
2             465.0  
3             539.0  
4             560.0  


In [7]:
# Having some overviews

print(df.info())
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   participant_id    120 non-null    int64  
 1   group             120 non-null    str    
 2   age               120 non-null    float64
 3   stress_score      112 non-null    float64
 4   sleep_quality     112 non-null    float64
 5   reaction_time_ms  120 non-null    float64
dtypes: float64(4), int64(1), str(1)
memory usage: 5.8 KB
None
       participant_id         age  stress_score  sleep_quality  \
count      120.000000  120.000000    112.000000     112.000000   
mean        60.500000   29.841667     22.340179       6.150893   
std         34.785054    7.544819      6.234450       1.334639   
min          1.000000    9.000000      2.600000       2.800000   
25%         30.750000   24.750000     17.575000       5.175000   
50%         60.500000   30.500000     22.600000       6.100000   
75%   

In [8]:
# Handeling missing values

print(df.isnull().sum())

participant_id      0
group               0
age                 0
stress_score        8
sleep_quality       8
reaction_time_ms    0
dtype: int64


In [9]:
# Group-based mean imputation:

df["stress_score"] = df.groupby("group")["stress_score"]\
    .transform(lambda x: x.fillna(x.mean()))

df["sleep_quality"] = df.groupby("group")["sleep_quality"]\
    .transform(lambda x: x.fillna(x.mean()))

# Check
print(df.isnull().sum())

participant_id      0
group               0
age                 0
stress_score        0
sleep_quality       0
reaction_time_ms    0
dtype: int64


In [10]:
# Convert age to integer
df["age"] = df["age"].astype(int)

# Ensure plausible ranges
df = df[df["stress_score"].between(0, 40)]
df = df[df["sleep_quality"].between(1, 10)]

print(df.head())

   participant_id        group  age  stress_score  sleep_quality  \
0               1      Control   18          27.8           9.20   
1               2  Mindfulness   24          24.5           3.10   
2               3      Control   26          26.9           6.08   
3               4      Control   38          33.4           6.90   
4               5      Control   33          20.5           6.40   

   reaction_time_ms  
0             514.0  
1             600.0  
2             465.0  
3             539.0  
4             560.0  


In [11]:
# Group-wise Summary Statistics

# Mean values by group
group_summary = df.groupby("group")[
    ["stress_score", "sleep_quality", "reaction_time_ms"]
].mean()

print(group_summary)

# Multiple summary
group_stats = df.groupby("group").agg({
    "stress_score": ["mean", "std"],
    "sleep_quality": ["mean", "std"],
    "reaction_time_ms": ["mean", "std"],
    "age": ["mean"]
})

print(group_stats)

             stress_score  sleep_quality  reaction_time_ms
group                                                     
Control         22.032609       6.080000        549.740000
Mindfulness     22.227800       6.188341        543.956522
            stress_score           sleep_quality           reaction_time_ms  \
                    mean       std          mean       std             mean   
group                                                                         
Control        22.032609  5.342439      6.080000  1.232817       549.740000   
Mindfulness    22.227800  5.931967      6.188341  1.341480       543.956522   

                              age  
                   std       mean  
group                              
Control      83.333559  29.600000  
Mindfulness  81.966176  29.956522  


In [12]:
# Be ready for visualization
group_stats.columns = [
    '_'.join(col).strip()
    for col in group_stats.columns.values
]

group_stats = group_stats.reset_index()

print(group_stats)

         group  stress_score_mean  stress_score_std  sleep_quality_mean  \
0      Control          22.032609          5.342439            6.080000   
1  Mindfulness          22.227800          5.931967            6.188341   

   sleep_quality_std  reaction_time_ms_mean  reaction_time_ms_std   age_mean  
0           1.232817             549.740000             83.333559  29.600000  
1           1.341480             543.956522             81.966176  29.956522  
